In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statistics import NormalDist, pstdev
import math
from collections import deque

In [1]:
class InventorySimulation:
    def __init__(self, initial_inventory, demand_series, lead_time = 0, review_period = 0,  
                        service_level = 0.95, forecast_series=None):
        """Các tham số đầu vào cho mô phỏng tồn kho.
        - demand_series: lịch sử (history) theo kỳ t=0..T-1
        - forecast_series: dự báo tương lai (có thể thiếu, lớp sẽ chỉ lấy vừa đủ)
        """
        self.initial_inventory = initial_inventory
        self.demand_series = list(demand_series)
        self.forecast_series = [] if forecast_series is None else list(forecast_series)
        self.lead_time = int(lead_time)
        self.review_period = int(review_period)
        self.service_level = float(service_level)

    # Helper: ghép history + forecast sao cho đủ tối thiểu need_len phần tử
    def _full_series(self, need_len: int):
        """
        Trả về list demand đủ độ dài >= need_len bằng cách nối demand_series + forecast_series.
        Nếu vẫn thiếu thì raise ValueError với thông điệp rõ ràng.
        """
        full = self.demand_series + self.forecast_series
        if len(full) < need_len:
            lack = need_len - len(full)
            raise ValueError(
                f"demand_series + forecast_series không đủ độ dài: cần >= {need_len}, hiện có {len(full)} (thiếu {lack}). "
                f"Hãy truyền thêm forecast_series hoặc kéo dài dự báo."
            )
        return full

    #Tính toán mức tồn kho an toàn trong từng thời điểm
    def calculate_safety_stock_series(self, risk_type = 'R+L'):
        """
        Trả về list SS_t theo công thức Greasley (2013):
        SS_t = Z * sigma_D_t * sqrt(risk_length)
        - risk_type = 'R+L'  -> dùng R + L
        - risk_type = 'L'    -> dùng L
        - risk_type = k (int không âm) -> dùng trực tiếp k
        Ghi chú: sigma_D_t tính theo expanding window từ lịch sử (demand_series) đến thời điểm t.
        """
        # Xác định độ dài rủi ro
        if risk_type == 'R+L':
            risk_length_value = self.review_period + self.lead_time
        elif risk_type == 'L':
            risk_length_value = self.lead_time
        elif isinstance(risk_type, int) and risk_type >= 0:
            risk_length_value = risk_type
        else:
            raise ValueError("risk_type phải là 'R+L', 'L' hoặc số nguyên không âm.")

        # Nếu độ dài rủi ro = 0 thì SS luôn = 0
        if risk_length_value == 0:
            return [0.0 for _ in range(len(self.demand_series))]

        # Z = Phi^{-1}(TSL)
        Z = NormalDist().inv_cdf(self.service_level)

        ss_series = []
        # Tính sigma_D_t theo expanding window từ dữ liệu LỊCH SỬ đến t
        # (chỉ dùng demand_series – không dùng forecast để ước lượng độ lệch chuẩn lịch sử)
        for t in range(1, len(self.demand_series) + 1):
            sigma_t = pstdev(self.demand_series[:t])  # population stdev; 1 phần tử -> 0
            ss_t = Z * sigma_t * math.sqrt(risk_length_value)
            ss_series.append(ss_t)

        return ss_series

    def calculate_reorderPoint(self, risk_type='R+L'):
        """
        s_t = sum_{w=t}^{t+R+L} D̂_w + SS_t
        D̂_w lấy từ full_series = demand_series + forecast_series.

        Trả về: list s_t cho t = 0..T_hist-1 (T_hist = len(demand_series)).
        """
        T_hist = len(self.demand_series)
        horizon = self.review_period + self.lead_time  # R + L

        # Cần đủ đến chỉ số t+R+L tại t = T_hist-1 => tổng độ dài tối thiểu:
        need_len = T_hist + horizon
        full = self._full_series(need_len)

        ss_series = self.calculate_safety_stock_series(risk_type=risk_type)

        s_series = []
        for t in range(T_hist):
            # t..t+R+L (bao gồm đầu-cuối) -> slice t : t+horizon+1
            exp_sum = sum(full[t : t + horizon + 1])
            s_t = exp_sum + ss_series[t]
            s_series.append(s_t)

        return s_series

    # giữ nguyên tên hàm cũ để không phá API, nhưng bạn có thể đổi thành calculate_orderUpto nếu muốn
    def calcualte_orderUpto(self, reorder_points=None, m=None, risk_type='R+L'):
        """
        Tính S_t theo (4):
            S_t = max( s_t , sum_{w=t}^{t+m} D̂_w )
        - D̂_w lấy từ full_series (history + forecast).
        - m: số kỳ được bao phủ; mặc định dùng (review_period + lead_time).

        Trả về: list S_t cho t = 0..T_hist-1
        """
        T_hist = len(self.demand_series)
        m = self.review_period + self.lead_time if m is None else int(m)

        need_len = T_hist + m
        full = self._full_series(need_len)

        if reorder_points is None:
            reorder_points = self.calculate_reorderPoint(risk_type=risk_type)

        S_series = []
        for t in range(T_hist):
            cover_sum = sum(full[t : t + m + 1])  # t..t+m
            S_t = max(reorder_points[t], cover_sum)
            S_series.append(S_t)

        return S_series

    def calculate_orderQuantity(self, reorder_points=None, order_uptos=None, risk_type='R+L'):
        """
        Tính Q_t theo (7):
            Q_t = S_t - I_t, nếu I_t <= s_t
                  0        , nếu I_t  > s_t
        - I_t: inventory position tại đầu kỳ t (on-hand + on-order).
        - Lost sales: không có backorder.
        - Đặt hàng đến S_t, hàng về sau L kỳ.

        Trả về: (order_qty_series, onhand_series, inventory_position_series)
        """
        T_hist = len(self.demand_series)
        L = self.lead_time

        if reorder_points is None:
            reorder_points = self.calculate_reorderPoint(risk_type=risk_type)
        if order_uptos is None:
            order_uptos = self.calcualte_orderUpto(reorder_points=reorder_points, risk_type=risk_type)

        on_hand = float(self.initial_inventory)
        pipeline = deque([0.0] * L, maxlen=L)
        orders, onhands, invpos = [], [], []

        for t in range(T_hist):
            # hàng về đầu kỳ
            if L > 0:
                on_hand += pipeline.popleft()

            # inventory position trước khi đặt hàng
            I_t = on_hand + (sum(pipeline) if L > 0 else 0.0)
            s_t = reorder_points[t]
            S_t = order_uptos[t]

            # quyết định đặt hàng
            if I_t <= s_t:
                Q_t = max(S_t - I_t, 0.0)
            else:
                Q_t = 0.0

            # đưa đơn hàng vào pipeline
            if L > 0:
                pipeline.append(Q_t)
            else:
                on_hand += Q_t  # L=0: hàng về ngay

            # phát sinh nhu cầu (lấy từ lịch sử ở thời điểm t)
            demand_t = float(self.demand_series[t])
            sales = min(on_hand, demand_t)
            on_hand -= sales  # lost sales if thiếu

            orders.append(Q_t)
            onhands.append(on_hand)
            I_t_after = on_hand + (sum(pipeline) if L > 0 else 0.0)
            invpos.append(I_t_after)

        return orders, onhands, invpos

    def calculate_cost(
        self,
        price: float,
        h: float,
        b: float,
        k: float,
        risk_type: str = 'R+L',
        reorder_points=None,
        order_uptos=None,
        return_breakdown: bool = False
    ):
        """
        Tính tổng chi phí bình quân mỗi kỳ:
            C_Tot = h/T * Σ I_t * p + b/T * Σ LS_t * p + k/T * Σ NO_t
        """
        T_hist = len(self.demand_series)
        L = self.lead_time

        if reorder_points is None:
            reorder_points = self.calculate_reorderPoint(risk_type=risk_type)
        if order_uptos is None:
            order_uptos = self.calcualte_orderUpto(reorder_points=reorder_points, risk_type=risk_type)

        on_hand = float(self.initial_inventory)
        pipeline = deque([0.0] * L, maxlen=L)

        sum_Ip = 0.0
        sum_LSp = 0.0
        sum_NO = 0

        per_period = {
            "I_t": [], "LS_t": [], "NO_t": [], "Q_t": [], "on_hand_end": []
        }

        for t in range(T_hist):
            if L > 0:
                on_hand += pipeline.popleft()

            I_pos = on_hand + (sum(pipeline) if L > 0 else 0.0)
            s_t = reorder_points[t]
            S_t = order_uptos[t]

            if I_pos <= s_t:
                Q_t = max(S_t - I_pos, 0.0)
                NO_t = 1
            else:
                Q_t = 0.0
                NO_t = 0

            if L > 0:
                pipeline.append(Q_t)
            else:
                on_hand += Q_t

            demand_t = float(self.demand_series[t])
            sales_t = min(on_hand, demand_t)
            LS_t = max(demand_t - sales_t, 0.0)
            on_hand -= sales_t

            I_t = max(on_hand, 0.0)
            sum_Ip  += I_t * price
            sum_LSp += LS_t * price
            sum_NO  += NO_t

            per_period["I_t"].append(I_t)
            per_period["LS_t"].append(LS_t)
            per_period["NO_t"].append(NO_t)
            per_period["Q_t"].append(Q_t)
            per_period["on_hand_end"].append(on_hand)

        C_H = (h / T_hist) * sum_Ip
        C_L = (b / T_hist) * sum_LSp
        C_O = (k / T_hist) * sum_NO
        C_Tot = C_H + C_L + C_O

        if return_breakdown:
            return {
                "avg_cost_per_period": C_Tot,
                "components": {"holding": C_H, "lost_sales": C_L, "ordering": C_O},
                "s_t": reorder_points,
                "S_t": order_uptos,
                "trace": per_period
            }
        return C_Tot
